# K-평균 기반 디지털 적응 수준 레이블 설계

목적: 전처리 완료 데이터에 K-평균 군집을 만들고, 각 군집을 1~4단계 디지털 적응 수준으로 해석한다.

진행 순서:
1. 학습/검증/테스트 데이터 불러오기
2. K-평균 입력 변수 정리
3. Q2K2, Q3 값 변환
4. 스케일링 후 K-평균(k=4) 실행
5. 군집별 특징 분석
6. 군집 번호를 디지털 단계 1~4단계로 매핑
7. 라벨이 포함된 데이터 저장


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 그래프에 한글이 깨지지 않도록 사용 가능한 한글 폰트를 설정한다.
available_fonts = {font.name for font in fm.fontManager.ttflist}
for korean_font in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if korean_font in available_fonts:
        plt.rcParams['font.family'] = korean_font
        break
plt.rcParams['axes.unicode_minus'] = False

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'preprocessed' / 'preprocessed'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'preprocessed' / 'labeled'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', DATA_DIR)
print('라벨 결과 저장 폴더:', OUTPUT_DIR)


## 1. 전처리 데이터 불러오기

K-평균은 전체 데이터 기준으로 군집을 만들기 위해 학습/검증/테스트 데이터를 먼저 합친다. 나중에 다시 나눌 수 있도록 `split` 컬럼을 추가한다.


In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
val = pd.read_csv(DATA_DIR / 'val.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

train['split'] = 'train'
val['split'] = 'val'
test['split'] = 'test'

df_all = pd.concat([train, val, test], ignore_index=True)
all_output_path = OUTPUT_DIR / 'all_with_split.csv'
df_all.to_csv(all_output_path, index=False, encoding='utf-8-sig')

print('저장 완료:', all_output_path)
print('전체 데이터 크기:', df_all.shape)
print('\nsplit별 개수')
print(df_all['split'].value_counts())
print('\nGROUP별 개수')
print(df_all['GROUP'].value_counts())
print('\n결측치 수:', df_all.isnull().sum().sum())


## 2. K-평균 입력 변수 준비와 값 정리

K-평균에는 인구통계 변수보다 디지털 행동과 적응 수준을 보여주는 변수를 사용한다.

저장 파일:
- `all_with_split.csv`: 학습/검증/테스트 데이터를 합친 파일
- `kmeans_metadata.csv`: 나중에 해석할 때 사용할 제외 변수 파일
- `kmeans_features_raw.csv`: K-평균 후보 변수 원본 파일
- `kmeans_features_clean.csv`: Q2K2/Q3을 1/0으로 변환한 뒤의 K-평균 입력 변수 파일

변환 규칙:
- `Q2K2_1`, `Q2K2_2`: 1=보유, 2=미보유 -> 1=보유, 0=미보유
- `Q3`: 1=인터넷 이용 가능, 2=인터넷 이용 불가 -> 1=이용 가능, 0=이용 불가


In [ ]:
exclude_cols = [
    'GROUP', 'YEAR', 'split',
    '연령',          # 연령
    '성별',          # 성별
    '직업',          # 직업
    '학력',          # 학력
    '가구구성형태',  # 가구구성형태
    '가구소득',      # 가구소득
    '거주지역',      # 거주지역
]

existing_exclude_cols = [c for c in exclude_cols if c in df_all.columns]
missing_exclude_cols = [c for c in exclude_cols if c not in df_all.columns]

metadata = df_all[existing_exclude_cols].copy()
X_raw = df_all.drop(columns=existing_exclude_cols)

metadata_path = OUTPUT_DIR / 'kmeans_metadata.csv'
raw_feature_path = OUTPUT_DIR / 'kmeans_features_raw.csv'
metadata.to_csv(metadata_path, index=False, encoding='utf-8-sig')
X_raw.to_csv(raw_feature_path, index=False, encoding='utf-8-sig')

df_cluster = X_raw.copy()

print('변환 전')
for col in ['Q2K2_1', 'Q2K2_2', 'Q3']:
    if col in df_cluster.columns:
        print(col, df_cluster[col].value_counts(dropna=False).sort_index().to_dict())

# Q2K2: 1=보유, 2=미보유 -> 1/0
for col in ['Q2K2_1', 'Q2K2_2']:
    if col in df_cluster.columns:
        df_cluster[col] = df_cluster[col].map({1: 1, 1.0: 1, 2: 0, 2.0: 0})

# Q3: 1=인터넷 이용 가능, 2=인터넷 이용 불가 -> 1/0
if 'Q3' in df_cluster.columns:
    df_cluster['Q3'] = df_cluster['Q3'].map({1: 1, 1.0: 1, 2: 0, 2.0: 0})

clean_feature_path = OUTPUT_DIR / 'kmeans_features_clean.csv'
df_cluster.to_csv(clean_feature_path, index=False, encoding='utf-8-sig')

X = df_cluster.copy()
feature_cols = X.columns.tolist()

print('\n변환 후')
for col in ['Q2K2_1', 'Q2K2_2', 'Q3']:
    if col in X.columns:
        print(col, X[col].value_counts(dropna=False).sort_index().to_dict())

print('\n원본 후보 변수 저장:', raw_feature_path)
print('정리된 입력 변수 저장:', clean_feature_path)
print('메타데이터 저장:', metadata_path)
print('제외한 컬럼:', existing_exclude_cols)
print('데이터에 없는 제외 후보 컬럼:', missing_exclude_cols)
print('K-평균 입력 변수 수:', len(feature_cols))
print('K-평균 입력 데이터 크기:', X.shape)
print('결측치 수:', X.isnull().sum().sum())
print('숫자가 아닌 컬럼:', X.select_dtypes(exclude='number').columns.tolist())


## 3. 스케일링

K-평균은 거리 기반 알고리즘이므로, 군집화 전에 모든 입력 변수를 표준화한다.

저장 파일:
- `kmeans_features_scaled.csv`: 표준화된 K-평균 입력 변수 파일
- `kmeans_scaler_params.csv`: 재현을 위한 변수별 평균, 스케일, 분산 파일


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
scaler_params = pd.DataFrame({
    'feature': X.columns,
    'mean': scaler.mean_,
    'scale': scaler.scale_,
    'var': scaler.var_,
})

scaled_feature_path = OUTPUT_DIR / 'kmeans_features_scaled.csv'
scaler_params_path = OUTPUT_DIR / 'kmeans_scaler_params.csv'
X_scaled_df.to_csv(scaled_feature_path, index=False, encoding='utf-8-sig')
scaler_params.to_csv(scaler_params_path, index=False, encoding='utf-8-sig')

print('스케일링된 입력 변수 저장:', scaled_feature_path)
print('스케일러 파라미터 저장:', scaler_params_path)
print('스케일링된 데이터 크기:', X_scaled_df.shape)
print('결측치 수:', X_scaled_df.isnull().sum().sum())
print('평균 절댓값 최댓값:', X_scaled_df.mean().abs().max())
print('표준편차 최솟값:', X_scaled_df.std(ddof=0).min())
print('표준편차 최댓값:', X_scaled_df.std(ddof=0).max())


## 4. K-평균 실행

프로젝트에서 디지털 적응 수준을 네 단계로 정의했기 때문에 `k=4`로 군집을 만든다.


In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=RANDOM_STATE,
    n_init=20
)

df_all['cluster'] = kmeans.fit_predict(X_scaled)
if 'row_id' not in df_all.columns:
    df_all.insert(0, 'row_id', df_all.index)

# 전체 3만여 행으로 실루엣 점수를 계산하면 오래 걸리므로 고정 크기 표본을 사용한다.
silhouette = silhouette_score(
    X_scaled,
    df_all['cluster'],
    sample_size=5000,
    random_state=RANDOM_STATE,
)

clustered_path = OUTPUT_DIR / 'all_with_cluster.csv'
labels_path = OUTPUT_DIR / 'kmeans_labels.csv'
metrics_path = OUTPUT_DIR / 'kmeans_metrics.csv'
centers_path = OUTPUT_DIR / 'kmeans_cluster_centers_scaled.csv'

df_all.to_csv(clustered_path, index=False, encoding='utf-8-sig')
df_all[['row_id', 'split', 'cluster']].to_csv(labels_path, index=False, encoding='utf-8-sig')
pd.DataFrame([{
    'k': 4,
    'inertia': kmeans.inertia_,
    'silhouette_sample_5000': silhouette,
    'random_state': RANDOM_STATE,
    'n_init': 20,
}]).to_csv(metrics_path, index=False, encoding='utf-8-sig')
pd.DataFrame(kmeans.cluster_centers_, columns=feature_cols).to_csv(
    centers_path,
    index_label='cluster',
    encoding='utf-8-sig',
)

print('군집별 개수')
print(df_all['cluster'].value_counts().sort_index())
print('\n관성값:', kmeans.inertia_)
print('실루엣 점수(표본 5000개):', silhouette)
print('\n저장 완료:', clustered_path)
print('저장 완료:', labels_path)
print('저장 완료:', metrics_path)
print('저장 완료:', centers_path)


## 5. 군집 특징 분석

각 군집에 대해 표본 수, 주요 변수 평균, 영역별 평균, 군집 구분에 크게 기여하는 변수, 메타데이터 분포를 정리한다.

`digital_activity_index`는 군집 해석을 돕기 위한 보조 지표다. 팀에서 승인하기 전까지는 최종 점수로 해석하지 않는다.


In [ ]:
analysis_dir = OUTPUT_DIR / 'cluster_analysis'
analysis_dir.mkdir(parents=True, exist_ok=True)

clusters = df_all['cluster']

cluster_counts = clusters.value_counts().sort_index().rename_axis('cluster').reset_index(name='count')
cluster_counts['percent'] = cluster_counts['count'] / len(clusters) * 100
cluster_counts.to_csv(analysis_dir / 'cluster_counts.csv', index=False, encoding='utf-8-sig')

core_cols = [
    'Q1_1', 'Q1_2', 'Q2K2_1', 'Q2K2_2', 'Q3', 'Q4B_1_1', 'Q4B_2_1',
    'Q4C_1', 'Q4C_2', 'Q10', 'Q11_1', 'Q11_2', 'Q11_3',
    'AI_인지', 'AI_사용빈도', 'AI_도움정도'
]
core_cols = [c for c in core_cols if c in X.columns]
cluster_core_summary = X.assign(cluster=clusters).groupby('cluster')[core_cols].mean().round(3)
cluster_core_summary.to_csv(analysis_dir / 'cluster_core_summary.csv', encoding='utf-8-sig')

prefix_groups = {
    'device_access_q2_q4': ['Q2K2', 'Q3', 'Q4B', 'Q4C'],
    'basic_skill_q5_q10': ['Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10'],
    'usage_time_q11': ['Q11'],
    'service_usage_q12_q19': ['Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19'],
    'difficulty_or_barrier_q20_q26': ['Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26'],
    'ai_nonuse_reason_q28': ['Q28'],
    'ai_related_q27_derived': ['AI_'],
    'digital_policy_need_q29_q31': ['Q29', 'Q30', 'Q31'],
}

def cols_by_prefix(prefixes):
    return [c for c in X.columns if any(str(c).startswith(prefix) for prefix in prefixes)]

domain_rows = []
for cluster in sorted(clusters.unique()):
    row = {'cluster': cluster, 'count': int((clusters == cluster).sum())}
    subset = X.loc[clusters == cluster]
    subset_scaled = X_scaled_df.loc[clusters == cluster]
    for name, prefixes in prefix_groups.items():
        cols = cols_by_prefix(prefixes)
        if cols:
            row[f'{name}_raw_mean'] = float(subset[cols].mean().mean())
            row[f'{name}_scaled_mean'] = float(subset_scaled[cols].mean().mean())
    domain_rows.append(row)

cluster_domain_summary = pd.DataFrame(domain_rows).round(3)
cluster_domain_summary.to_csv(analysis_dir / 'cluster_domain_summary.csv', index=False, encoding='utf-8-sig')

scaled_means = X_scaled_df.assign(cluster=clusters).groupby('cluster').mean()
scaled_means.to_csv(analysis_dir / 'cluster_scaled_feature_means.csv', encoding='utf-8-sig')

top_rows = []
for cluster, row in scaled_means.iterrows():
    for rank, (feature, value) in enumerate(row.sort_values(ascending=False).head(15).items(), start=1):
        top_rows.append({'cluster': cluster, 'direction': 'high', 'rank': rank, 'feature': feature, 'scaled_mean': value})
    for rank, (feature, value) in enumerate(row.sort_values(ascending=True).head(15).items(), start=1):
        top_rows.append({'cluster': cluster, 'direction': 'low', 'rank': rank, 'feature': feature, 'scaled_mean': value})
cluster_top_features = pd.DataFrame(top_rows)
cluster_top_features['scaled_mean'] = cluster_top_features['scaled_mean'].round(3)
cluster_top_features.to_csv(analysis_dir / 'cluster_top_features.csv', index=False, encoding='utf-8-sig')

for col in ['GROUP', 'YEAR', 'split']:
    counts = pd.crosstab(metadata[col], clusters)
    pct = pd.crosstab(metadata[col], clusters, normalize='columns') * 100
    counts.to_csv(analysis_dir / f'cluster_{col}_counts.csv', encoding='utf-8-sig')
    pct.round(2).to_csv(analysis_dir / f'cluster_{col}_pct_by_cluster.csv', encoding='utf-8-sig')

rank_cols = [
    'device_access_q2_q4_scaled_mean',
    'basic_skill_q5_q10_scaled_mean',
    'service_usage_q12_q19_scaled_mean',
    'ai_related_q27_derived_scaled_mean',
]
rank_cols = [c for c in rank_cols if c in cluster_domain_summary.columns]
cluster_rank = cluster_domain_summary[['cluster', 'count'] + rank_cols].copy()
cluster_rank['digital_activity_index'] = cluster_rank[rank_cols].mean(axis=1)
cluster_rank = cluster_rank.sort_values('digital_activity_index').reset_index(drop=True)
cluster_rank['activity_order_low_to_high'] = np.arange(1, len(cluster_rank) + 1)
cluster_rank.round(3).to_csv(analysis_dir / 'cluster_activity_ranking_helper.csv', index=False, encoding='utf-8-sig')

plt.figure(figsize=(7, 4))
sns.barplot(data=cluster_counts, x='cluster', y='count', color='#4C78A8')
plt.title('군집별 표본 수')
plt.xlabel('군집')
plt.ylabel('표본 수')
plt.tight_layout()
plt.savefig(analysis_dir / 'cluster_counts.png', dpi=160, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 4))
sns.barplot(data=cluster_rank, x='cluster', y='digital_activity_index', order=cluster_rank['cluster'], color='#59A14F')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('군집별 디지털 활동 지표')
plt.xlabel('군집')
plt.ylabel('디지털 활동 지표')
plt.tight_layout()
plt.savefig(analysis_dir / 'cluster_activity_index.png', dpi=160, bbox_inches='tight')
plt.show()

scaled_cols = [c for c in cluster_domain_summary.columns if c.endswith('_scaled_mean')]
heat = cluster_domain_summary.set_index('cluster')[scaled_cols]
domain_label_map = {
    'device_access_q2_q4': '기기 접근성',
    'basic_skill_q5_q10': '기본 역량',
    'usage_time_q11': '이용 시간',
    'service_usage_q12_q19': '서비스 이용',
    'difficulty_or_barrier_q20_q26': '어려움/장벽',
    'ai_nonuse_reason_q28': 'AI 미이용 이유',
    'ai_related_q27_derived': 'AI 관련',
    'digital_policy_need_q29_q31': '정책/교육 필요',
}
heat.columns = [domain_label_map.get(c.replace('_scaled_mean', ''), c.replace('_scaled_mean', '')) for c in heat.columns]
plt.figure(figsize=(12, 4.8))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('군집별 영역 요약(스케일링 평균)')
plt.tight_layout()
plt.savefig(analysis_dir / 'cluster_domain_scaled_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

group_pct = pd.read_csv(analysis_dir / 'cluster_GROUP_pct_by_cluster.csv', index_col=0)
group_pct_display = group_pct.copy()
group_pct_display.to_csv(analysis_dir / 'cluster_GROUP_pct_by_cluster_kr.csv', encoding='utf-8-sig')
plt.figure(figsize=(7, 4.5))
sns.heatmap(group_pct_display, annot=True, fmt='.1f', cmap='Blues')
plt.title('각 군집 내 집단 비율(%)')
plt.xlabel('군집')
plt.ylabel('집단')
plt.tight_layout()
plt.savefig(analysis_dir / 'cluster_group_pct_heatmap_kr.png', dpi=160, bbox_inches='tight')
plt.show()

print('분석 결과 폴더 저장 완료:', analysis_dir)
display(cluster_counts.rename(columns={'cluster': '군집', 'count': '표본 수', 'percent': '비율'}))
display(cluster_core_summary)
display(cluster_domain_summary)
display(cluster_rank.round(3))
display(cluster_top_features.head(30))


## 6. 디지털 단계 매핑

아래 매핑은 군집별 특징표를 보고 직접 수정해야 한다.

예시:

```python
stage_map = {
    2: 1,  # 완전 소외
    0: 2,  # 제한적 이용
    3: 3,  # 부분 적응
    1: 4,  # 자립 적응
}
```


In [ ]:
# 군집 활동 순위 표를 기준으로 매핑한다.
# 디지털 활동이 낮을수록 낮은 디지털 단계로 매핑한다.
stage_map = {
    2: 1,  # 완전 소외 / 가장 낮은 활동 수준
    0: 2,  # 제한적 이용
    1: 3,  # 부분 적응
    3: 4,  # 자립 적응 / 가장 높은 활동 수준
}

df_all['digital_stage'] = df_all['cluster'].map(stage_map)
df_all['digital_stage_name'] = df_all['digital_stage'].map({
    1: '1단계_완전_소외',
    2: '2단계_제한적_이용',
    3: '3단계_부분_적응',
    4: '4단계_자립_적응',
})

stage_mapping_path = OUTPUT_DIR / 'cluster_to_stage_mapping.csv'
pd.DataFrame([
    {'cluster': cluster, 'digital_stage': stage, 'digital_stage_name': df_all.loc[df_all['cluster'].eq(cluster), 'digital_stage_name'].iloc[0]}
    for cluster, stage in stage_map.items()
]).sort_values('digital_stage').to_csv(stage_mapping_path, index=False, encoding='utf-8-sig')

stage_output_path = OUTPUT_DIR / 'all_with_stage.csv'
stage_label_path = OUTPUT_DIR / 'kmeans_stage_labels.csv'
df_all.to_csv(stage_output_path, index=False, encoding='utf-8-sig')
df_all[['row_id', 'split', 'cluster', 'digital_stage', 'digital_stage_name']].to_csv(
    stage_label_path,
    index=False,
    encoding='utf-8-sig',
)

print('digital_stage 분포')
print(df_all['digital_stage'].value_counts().sort_index())
print('\n저장 완료:', stage_mapping_path)
print('저장 완료:', stage_output_path)
print('저장 완료:', stage_label_path)


## 7. 디지털 단계 검증 시각화

매핑된 디지털 단계가 의도대로 작동하는지 확인한다. 핵심 확인 기준은 1단계에서 4단계로 갈수록 디지털 활동 수준이 높아지는지 여부다.


In [ ]:
validation_dir = OUTPUT_DIR / 'stage_validation'
validation_dir.mkdir(parents=True, exist_ok=True)

viz_df = df_all[['row_id', 'GROUP', 'YEAR', 'split', 'cluster', 'digital_stage', 'digital_stage_name']].copy()

validation_prefix_groups = {
    'device_access': ['Q2K2', 'Q3', 'Q4B', 'Q4C'],
    'basic_skill': ['Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10'],
    'usage_time': ['Q11'],
    'service_usage': ['Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19'],
    'difficulty_or_barrier': ['Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26'],
    'ai_related': ['AI_'],
    'digital_policy_need': ['Q29', 'Q30', 'Q31'],
}

for score_name, prefixes in validation_prefix_groups.items():
    cols = [c for c in X_scaled_df.columns if any(str(c).startswith(prefix) for prefix in prefixes)]
    if cols:
        viz_df[f'{score_name}_score'] = X_scaled_df[cols].mean(axis=1)

activity_score_cols = [
    'device_access_score',
    'basic_skill_score',
    'service_usage_score',
    'ai_related_score',
]
activity_score_cols = [c for c in activity_score_cols if c in viz_df.columns]
viz_df['digital_activity_score'] = viz_df[activity_score_cols].mean(axis=1)

score_cols = [c for c in viz_df.columns if c.endswith('_score')]
ordered_score_cols = []
for col in ['digital_activity_score'] + score_cols:
    if col not in ordered_score_cols:
        ordered_score_cols.append(col)

stage_summary = (
    viz_df.groupby(['digital_stage', 'digital_stage_name'])[ordered_score_cols]
    .mean()
    .round(3)
    .reset_index()
)
stage_summary.to_csv(validation_dir / 'stage_score_summary.csv', index=False, encoding='utf-8-sig')

stage_counts = (
    viz_df['digital_stage']
    .value_counts()
    .sort_index()
    .rename_axis('digital_stage')
    .reset_index(name='count')
)
stage_counts['percent'] = stage_counts['count'] / len(viz_df) * 100
stage_counts['digital_stage_name'] = stage_counts['digital_stage'].map(
    viz_df.drop_duplicates('digital_stage').set_index('digital_stage')['digital_stage_name']
)
stage_counts.to_csv(validation_dir / 'stage_counts.csv', index=False, encoding='utf-8-sig')

stage_group_counts = pd.crosstab(viz_df['GROUP'], viz_df['digital_stage'])
stage_group_pct = pd.crosstab(viz_df['GROUP'], viz_df['digital_stage'], normalize='columns') * 100
stage_group_counts.to_csv(validation_dir / 'stage_GROUP_counts.csv', encoding='utf-8-sig')
stage_group_pct.round(2).to_csv(validation_dir / 'stage_GROUP_pct_by_stage.csv', encoding='utf-8-sig')
pd.crosstab(viz_df['YEAR'], viz_df['digital_stage']).to_csv(validation_dir / 'stage_YEAR_counts.csv', encoding='utf-8-sig')
viz_df.to_csv(validation_dir / 'stage_validation_scores.csv', index=False, encoding='utf-8-sig')

def savefig(name):
    plt.tight_layout()
    plt.savefig(validation_dir / name, dpi=160, bbox_inches='tight')
    plt.show()

plt.figure(figsize=(7, 4))
sns.barplot(data=stage_counts, x='digital_stage', y='count', color='#4C78A8')
plt.title('디지털 단계별 표본 수')
plt.xlabel('디지털 단계')
plt.ylabel('표본 수')
savefig('stage_counts_bar.png')

plt.figure(figsize=(6, 6))
plt.pie(stage_counts['count'], labels=stage_counts['digital_stage'], autopct='%1.1f%%', startangle=90)
plt.title('디지털 단계 분포')
savefig('stage_distribution_pie.png')

plt.figure(figsize=(8, 4.5))
sns.boxplot(data=viz_df, x='digital_stage', y='digital_activity_score', color='#59A14F')
plt.title('디지털 단계별 디지털 활동 점수')
plt.xlabel('디지털 단계')
plt.ylabel('평균 스케일링 활동 점수')
savefig('stage_digital_activity_boxplot.png')

plt.figure(figsize=(8, 4.5))
sns.boxplot(data=viz_df, x='digital_stage', y='service_usage_score', color='#F28E2B')
plt.title('디지털 단계별 서비스 이용 점수')
plt.xlabel('디지털 단계')
plt.ylabel('평균 스케일링 서비스 이용 점수')
savefig('stage_service_usage_boxplot.png')

if 'ai_related_score' in viz_df.columns:
    plt.figure(figsize=(8, 4.5))
    sns.boxplot(data=viz_df, x='digital_stage', y='ai_related_score', color='#B07AA1')
    plt.title('디지털 단계별 AI 관련 점수')
    plt.xlabel('디지털 단계')
    plt.ylabel('평균 스케일링 AI 점수')
    savefig('stage_ai_related_boxplot.png')

heat = stage_summary.set_index('digital_stage')[ordered_score_cols]
score_label_map = {
    'digital_activity_score': '디지털 활동',
    'device_access_score': '기기 접근성',
    'basic_skill_score': '기본 역량',
    'usage_time_score': '이용 시간',
    'service_usage_score': '서비스 이용',
    'difficulty_or_barrier_score': '어려움/장벽',
    'ai_related_score': 'AI 관련',
    'digital_policy_need_score': '정책/교육 필요',
}
heat.columns = [score_label_map.get(c, c) for c in heat.columns]
plt.figure(figsize=(10, 4.8))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('디지털 단계별 점수 요약')
plt.xlabel('점수')
plt.ylabel('디지털 단계')
savefig('stage_score_heatmap.png')

stage_group_pct_kr = stage_group_pct.copy()
stage_group_pct_kr.round(2).to_csv(validation_dir / 'stage_GROUP_pct_by_stage_kr.csv', encoding='utf-8-sig')
plt.figure(figsize=(7, 4.5))
sns.heatmap(stage_group_pct_kr, annot=True, fmt='.1f', cmap='Blues')
plt.title('각 디지털 단계 내 집단 비율(%)')
plt.xlabel('디지털 단계')
plt.ylabel('집단')
savefig('stage_group_pct_heatmap_kr.png')

print('검증 결과 폴더 저장 완료:', validation_dir)
display(stage_counts.rename(columns={'digital_stage': '디지털 단계', 'count': '표본 수', 'percent': '비율'}))
display(stage_summary)
display(stage_group_pct.round(2))


## 8. 최종 라벨 학습/검증/테스트 파일 저장

원래 분할 기준에 맞춰 라벨이 포함된 파일을 각각 저장한다. 이후 분류 모델에서는 `digital_stage`를 예측 목표로 사용한다.


In [ ]:
required_cols = ['split', 'cluster', 'digital_stage', 'digital_stage_name']
missing = [c for c in required_cols if c not in df_all.columns]
if missing:
    raise ValueError(f'필수 컬럼이 없습니다: {missing}')

output_paths = {}
for split_name in ['train', 'val', 'test']:
    out = df_all[df_all['split'] == split_name].copy()
    out = out.drop(columns=['split'])
    output_path = OUTPUT_DIR / f'{split_name}_labeled.csv'
    out.to_csv(output_path, index=False, encoding='utf-8-sig')
    output_paths[split_name] = output_path

summary_rows = []
for split_name, path in output_paths.items():
    out = pd.read_csv(path)
    row = {
        'split': split_name,
        'path': str(path),
        'rows': len(out),
        'columns': out.shape[1],
        'missing': int(out.isna().sum().sum()),
    }
    for stage, count in out['digital_stage'].value_counts().sort_index().items():
        row[f'stage_{int(stage)}_count'] = int(count)
    summary_rows.append(row)

labeled_split_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / 'labeled_split_summary.csv'
labeled_split_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

print('라벨 파일 저장 완료:')
for split_name, path in output_paths.items():
    print(split_name, path)
print('\n요약 파일 저장 완료:', summary_path)
display(labeled_split_summary)


## 9. 참고 사항

- `digital_stage`는 최종 네 단계 타깃 라벨이다.
- `cluster`는 K-평균에서 나온 원래 군집 번호다.
- 이후 분류 모델 학습에는 `train_labeled.csv`, `val_labeled.csv`, `test_labeled.csv`를 사용한다.


In [ ]:
print('최종 예측 목표 컬럼: digital_stage')
print('최종 라벨 파일 저장 위치:', OUTPUT_DIR)
